> **Provenance.** This notebook is from Wen & Wang (2025), *Code for: Seismic wave propagation simulation with quantum computing*, Zenodo, [doi:10.5281/zenodo.14868887](https://doi.org/10.5281/zenodo.14868887), licensed [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Changes in this repository: outputs cleared, and this note plus the setup cell below added. The notebook uses the classical simulated-annealing sampler by default; the D-Wave QPU sampler is left commented out, as in the original.

In [ ]:
# Added for this repository: make the top-level `core` package importable
# when the notebook is run from the notebooks/ folder.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

In [ ]:
import numpy as np
from core import stagger_FD_coeff as sfci
from core.utils import cholesky_solver
from neal import SimulatedAnnealingSampler
from dwave.system import DWaveSampler
from core.QUBO_solvers import IterSampleSolver
from collections import ChainMap
from matplotlib.ticker import MultipleLocator
import matplotlib.pyplot as plt

pi = np.pi

In [ ]:
v = 2500
h = 10
tao = 0.001
freq_max_set = [100]
M_set = [10]
max_iter_num = 30

# sampler = DWaveSampler(token='',
#                        solver='Advantage_system6.2',
#                        region='na-west-1')
sampler = SimulatedAnnealingSampler()

tra_a0_dict = {}
for M in M_set:
    tra_a0_dict["Tra" + f" M={M}"] = sfci.Taylor_Coef(M)

wang_a0_dict = {}
for M, freq_max in zip(M_set, freq_max_set):
    linear_sys = sfci.Stagger_FD_coeff_3D(v, h, tao, freq_max)
    wang_a0_dict["Wang" + f" M={M}"] = linear_sys.solve(
        M=M,
        alpha=1.0,
        alpha_decay=0.1,
        beta=1.0,
        epsilon=1e-10,
        max_iter_num=max_iter_num,
        mode_of_construct="lstsq",
        solver=cholesky_solver,
        show_process=False,
    )

qa_a0_dict = {}
for M, freq_max in zip(M_set, freq_max_set):
    linear_sys = sfci.Stagger_FD_coeff_3D(v, h, tao, freq_max)
    sis = IterSampleSolver(
        sampler=sampler,
        x0=np.ones(M) / 100,  # initial vector of Delta a
        num_sample=100,  # 5000
        R=10,
        alpha = 0.1,
        fixed_point=1,
        epsilon=1e-5,
        maxiter=3,  # max iteration in small iteration
        resultMod="lowest_energy",
    )
    qa_a0 = linear_sys.solve(
        M=M,
        coeff_vec=sfci.Taylor_Coef(M),
        alpha=1.0,
        alpha_decay=0.1,
        beta=1.0,
        epsilon=1e-10,
        max_iter_num=max_iter_num,
        mode_of_construct="direct",
        solver=sis.solve,
        show_process=False,
    )
    qa_a0_dict[f"QA M={M}"] = qa_a0

result_dict = dict(ChainMap(tra_a0_dict, wang_a0_dict, qa_a0_dict))

In [ ]:
pi = np.pi
v = 2500
h = 10
tao = 0.001
max_iter_num = 20

# 创建王老师方法中定义的线性系统对象
linear_sys = sfci.Stagger_FD_coeff_3D(v, h, tao)

fig, ax = plt.subplots(1, 1, figsize=(5, 4), dpi=150)

lw = 1.1  # linwidth of box
style_set = ["-", "--", ":", "-."]
color_set = ["maroon", "midnightblue", "purple"]
linwith = 1.2
counter_s = 0
counter_c = 0
target = "QA"  # Tra, Wang, QA
for label, coeff in result_dict.items():
    for str_theta, theta in zip(
        ["0", "$\\pi/8$", "$\\pi/4$", "$3\\pi/8$"], [0, pi / 8, pi / 4, 3 * pi / 8]
    ):
        for str_phi, phi in zip(["0", "$\\pi/8$", "$\\pi/4$"], [0, pi / 8, pi / 4]):
            sigma = linear_sys.cal_sigma(
                v * tao / h,
                np.linspace(pi / 1000, pi, 1000),
                coeff,
                theta=theta,
                phi=phi,
            )
            if target in label:
                ax.plot(
                    np.linspace(pi / 1000, pi, 1000),
                    sigma,
                    color=color_set[counter_c],
                    linewidth=linwith,
                    ls=style_set[counter_s],
                    label=target
                    + "  "
                    + f"$\\theta=${str_theta}"
                    + ",  "
                    + f"$\\phi=${str_phi}",
                )

            ax.set_ylim([0.9, 1.05])
            ax.set_xlim([0, 3.142])
            ax.set_ylabel("$\delta(\\theta, \\phi)$", fontsize=12)
            ax.set_xlabel("$kh$", fontsize=12)
            ax.set_xticks([0, 0.5, 1, 1.5, 2, 2.5, 3])
            ax.set_yticks([0.9, 0.95, 1.0, 1.05])
            # 设置边框粗细
            ax.spines["left"].set_linewidth(lw)
            ax.spines["top"].set_linewidth(lw)
            ax.spines["right"].set_linewidth(lw)
            ax.spines["bottom"].set_linewidth(lw)

            # 设置主刻度格式
            ax.tick_params(
                top=True,
                right=True,
                which="major",
                direction="in",
                length=4,
                width=1.0,
                colors="black",
                labelsize=8,
            )
            # 设置次刻度格式
            ax.xaxis.set_minor_locator(
                MultipleLocator(0.1)
            )  # 显示子刻度,括号里的是间距
            ax.yaxis.set_minor_locator(MultipleLocator(0.01))
            # ax.xaxis.set_minor_locator(AutoMinorLocator()) #  MultipleLocator(0.2)
            ax.tick_params(
                top=True,
                right=True,
                direction="in",
                which="minor",
                length=2,
                width=0.8,
                colors="black",
            )

            ax.grid(
                visible=True, axis="both", which="major", linestyle="--", linewidth=0.5
            )
            ax.legend(
                title="",
                fontsize="xx-small",
                loc="lower left",
                borderpad=0.5,  # 图例内部留白
                borderaxespad=1.5,  # 图框边界距离坐标轴距离
                labelspacing=0.3,  # 图例条目间距
                edgecolor="black",
                handlelength=3.2,
            )

            counter_c = (counter_c + 1) % 3
        counter_s = (counter_s + 1) % 4

plt.show()
# fig.savefig(f'3D_reuslt_{target}.png', dpi=300)